In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Step 1: Install Flask and pyngrok
!pip install flask flask-ngrok


In [ ]:
!pip install pyngrok

In [ ]:
import torch
print("Torch version:",torch.__version__)
print("Is CUDA enabled?",torch.cuda.is_available())

In [ ]:
!pip install torchgeometry

In [ ]:
import argparse
import torch
from torch import nn
from torch.nn import functional as F
import torchgeometry as tgm
import time
import os
os.chdir('/content/drive/MyDrive/VDress/')
from module1 import VITONDataset, VITONDataLoader
from module2 import SegGenerator, GMM, ALIASGenerator
from module3 import gen_noise, load_checkpoint, save_images


def get_opt():

    parser = argparse.ArgumentParser()
    #parser.add_argument('--name', type=str, required=True)
    parser.add_argument('--name', type=str, default = 'output')
    parser.add_argument('-b', '--batch_size', type=int, default=1)
    parser.add_argument('-j', '--workers', type=int, default=1)
    parser.add_argument('--load_height', type=int, default=1024)
    parser.add_argument('--load_width', type=int, default=768)
    parser.add_argument('--shuffle', action='store_true')

    parser.add_argument('--dataset_dir', type=str, default='/content/drive/My Drive/VDress/realtime_data/')
    parser.add_argument('--dataset_mode', type=str, default='/content/drive/My Drive/VDress/realtime_data/test')
    parser.add_argument('--dataset_list', type=str, default='/content/drive/My Drive/VDress/realtime_data/test_pairs.txt')
    parser.add_argument('--checkpoint_dir', type=str, default='/content/drive/My Drive/VDress/checkpoints/')
    parser.add_argument('--save_dir', type=str, default='/content/drive/My Drive/VDress/Realtime_otput/')

    parser.add_argument('--display_freq', type=int, default=1)

    parser.add_argument('--seg_checkpoint', type=str, default='/content/drive/My Drive/VDress/checkpoints/seg_final.pth')
    parser.add_argument('--gmm_checkpoint', type=str, default='/content/drive/My Drive/VDress/checkpoints/gmm_final.pth')
    parser.add_argument('--alias_checkpoint', type=str, default='/content/drive/My Drive/VDress/checkpoints/alias_final.pth')

    # common
    parser.add_argument('--semantic_nc', type=int, default=13, help='# of human-parsing map classes')
    parser.add_argument('--init_type', choices=['normal', 'xavier', 'xavier_uniform', 'kaiming', 'orthogonal', 'none'], default='xavier')
    parser.add_argument('--init_variance', type=float, default=0.02, help='variance of the initialization distribution')

    # for GMM
    parser.add_argument('--grid_size', type=int, default=5)

    # for ALIASGenerator
    parser.add_argument('--norm_G', type=str, default='spectralaliasinstance')
    parser.add_argument('--ngf', type=int, default=64, help='# of generator filters in the first conv layer')
    parser.add_argument('--num_upsampling_layers', choices=['normal', 'more', 'most'], default='most',
                        help='If \'more\', add upsampling layer between the two middle resnet blocks. '
                             'If \'most\', also add one more (upsampling + resnet) layer at the end of the generator.')

    #opt = parser.parse_args()
    opt = parser.parse_args(args=[])
    #opt = vars(parser.parse_args())
    return opt


def test(opt, seg, gmm, alias):
    up = nn.Upsample(size=(opt.load_height, opt.load_width), mode='bilinear')
    gauss = tgm.image.GaussianBlur((15, 15), (3, 3))
    gauss.cuda()

    test_dataset = VITONDataset(opt)
    test_loader = VITONDataLoader(opt, test_dataset)

    with torch.no_grad():
        for i, inputs in enumerate(test_loader.data_loader):
            img_names = inputs['img_name']
            c_names = inputs['c_name']['unpaired']

            img_agnostic = inputs['img_agnostic'].cuda()
            parse_agnostic = inputs['parse_agnostic'].cuda()
            pose = inputs['pose'].cuda()
            c = inputs['cloth']['unpaired'].cuda()
            cm = inputs['cloth_mask']['unpaired'].cuda()

            # Part 1. Segmentation generation
            parse_agnostic_down = F.interpolate(parse_agnostic, size=(256, 192), mode='bilinear')
            pose_down = F.interpolate(pose, size=(256, 192), mode='bilinear')
            c_masked_down = F.interpolate(c * cm, size=(256, 192), mode='bilinear')
            cm_down = F.interpolate(cm, size=(256, 192), mode='bilinear')
            seg_input = torch.cat((cm_down, c_masked_down, parse_agnostic_down, pose_down, gen_noise(cm_down.size()).cuda()), dim=1)

            parse_pred_down = seg(seg_input)
            parse_pred = gauss(up(parse_pred_down))
            parse_pred = parse_pred.argmax(dim=1)[:, None]

            parse_old = torch.zeros(parse_pred.size(0), 13, opt.load_height, opt.load_width, dtype=torch.float).cuda()
            parse_old.scatter_(1, parse_pred, 1.0)

            labels = {
                0:  ['background',  [0]],
                1:  ['paste',       [2, 4, 7, 8, 9, 10, 11]],
                2:  ['upper',       [3]],
                3:  ['hair',        [1]],
                4:  ['left_arm',    [5]],
                5:  ['right_arm',   [6]],
                6:  ['noise',       [12]]
            }
            parse = torch.zeros(parse_pred.size(0), 7, opt.load_height, opt.load_width, dtype=torch.float).cuda()
            for j in range(len(labels)):
                for label in labels[j][1]:
                    parse[:, j] += parse_old[:, label]

            # Part 2. Clothes Deformation
            agnostic_gmm = F.interpolate(img_agnostic, size=(256, 192), mode='nearest')
            parse_cloth_gmm = F.interpolate(parse[:, 2:3], size=(256, 192), mode='nearest')
            pose_gmm = F.interpolate(pose, size=(256, 192), mode='nearest')
            c_gmm = F.interpolate(c, size=(256, 192), mode='nearest')
            gmm_input = torch.cat((parse_cloth_gmm, pose_gmm, agnostic_gmm), dim=1)

            _, warped_grid = gmm(gmm_input, c_gmm)
            warped_c = F.grid_sample(c, warped_grid, padding_mode='border')
            warped_cm = F.grid_sample(cm, warped_grid, padding_mode='border')

            # Part 3. Try-on synthesis
            misalign_mask = parse[:, 2:3] - warped_cm
            misalign_mask[misalign_mask < 0.0] = 0.0
            parse_div = torch.cat((parse, misalign_mask), dim=1)
            parse_div[:, 2:3] -= misalign_mask

            output = alias(torch.cat((img_agnostic, pose, warped_c), dim=1), parse, parse_div, misalign_mask)

            unpaired_names = []
            for img_name, c_name in zip(img_names, c_names):
                unpaired_names.append('{}_{}'.format(img_name.split('_')[0], c_name))

            save_images(output, unpaired_names, os.path.join(opt.save_dir, opt.name))

            if (i + 1) % opt.display_freq == 0:
                print("step: {}".format(i + 1))


def main():
    opt = get_opt()
    print(opt)

    if not os.path.exists(os.path.join(opt.save_dir, opt.name)):
        os.makedirs(os.path.join(opt.save_dir, opt.name))

    seg = SegGenerator(opt, input_nc=opt.semantic_nc + 8, output_nc=opt.semantic_nc)
    gmm = GMM(opt, inputA_nc=7, inputB_nc=3)
    opt.semantic_nc = 7
    alias = ALIASGenerator(opt, input_nc=9)
    opt.semantic_nc = 13

    load_checkpoint(seg, os.path.join(opt.checkpoint_dir, opt.seg_checkpoint))
    load_checkpoint(gmm, os.path.join(opt.checkpoint_dir, opt.gmm_checkpoint))
    load_checkpoint(alias, os.path.join(opt.checkpoint_dir, opt.alias_checkpoint))

    seg.cuda().eval()
    gmm.cuda().eval()
    alias.cuda().eval()
    test(opt, seg, gmm, alias)







In [ ]:
import shutil

In [ ]:
from pyngrok import ngrok
# Step 2: Import necessary libraries
from flask import Flask, request, render_template
from flask_ngrok import run_with_ngrok
import os
port_no = 5000
# Step 3: Initialize Flask app and ngrok
app = Flask(__name__,template_folder='/content/drive/My Drive/VDress/templates',static_folder='static')
#app = Flask(__name__,template_folder='/content/drive/My Drive/VDress/templates/',static_folder='/content/drive/My Drive/VDress/static')
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
public_url =  ngrok.connect(port_no).public_url
#run_with_ngrok(app)  # Initialize ngrok when app is run

# Step 4: Configure the upload folder and allowed extensions
#UPLOAD_FOLDER = '/content/drive/My Drive/flask'
#UPLOAD_FOLDER = '/content/drive/My Drive/VDress/realtime_data/test/cloth'
#UPLOAD_FOLDER2 = '/content/drive/My Drive/VDress/realtime_data/test/image'
#os.makedirs(UPLOAD_FOLDER, exist_ok=True)
#app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
#app.config['UPLOAD_FOLDER2'] = UPLOAD_FOLDER2
#ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'gif'}




from flask import Flask, render_template, request, url_for
import os
from werkzeug.utils import secure_filename
import shutil
import time



# Configure upload folder and allowed extensions
UPLOAD_FOLDER = '/content/drive/My Drive/VDress/static/uploads'
GRID_FOLDER = '/content/drive/My Drive/VDress/static/grid_images'
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'gif'}

# Ensure the upload folder exists
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
from google.colab import drive
import os
from flask import Flask, render_template, send_from_directory


# Check if the folder exists
assert os.path.exists('/content/drive/My Drive/VDress/static/grid_images'), "Folder does not exist!"

# Create a symlink to the static folder if it doesn't exist
if not os.path.exists('/static'):
    os.symlink('/content/drive/My Drive/VDress/static', '/static')

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

@app.route('/')
def index():
    # List images from the grid_images folder
    grid_images = os.listdir(GRID_FOLDER)

    print('grid_images',grid_images)
    '''
    for i in range(len(grid_images)):
      print('no of image: ',i)
      grid_images[i]=GRID_FOLDER+'/'+grid_images[i]
      #grid_images2=GRID_FOLDER+'/'+grid_images[i]
      print('name of images: ',grid_images)
      # <div>
      # <img src="{{ url_for('static', filename=image_url) }}"/>
      # </div>

    #return render_template('index.html', grid_images=grid_images2)
    print('grid_images....',grid_images)
    '''
    return render_template('index.html', grid_images=grid_images)


# def index():
#         path = '/content/drive/My Drive/VDress/static/grid_images'
#         uploads = sorted(os.listdir(path), key=lambda x: os.path.getctime(path+x))
#         print(uploads)
#         #uploads = os.listdir('static/uploads')
#         #uploads = [path+'/'+ file for file in uploads]
#         uploads = [path+'/'+ file for file in uploads]
#         print('kkkkk: ',uploads)
#         #uploads.reverse()
#         return render_template("index.html",uploads=uploads)

@app.route('/images/<filename>')
def serve_image(filename):
    return send_from_directory(GRID_FOLDER, filename)
@app.route('/rimages/<filename>')
def serve_rimages(filename):
    return send_from_directory(UPLOAD_FOLDER, filename)

@app.route('/upload', methods=['POST'])
def upload():
    # Get selected grid image
    selected_image = request.form.get('selected_image')
    selected_image_path = None

    # If a grid image was selected, copy it to the uploads folder
    if selected_image:
        original_path = os.path.join(GRID_FOLDER, selected_image)
        selected_image_filename = f"selected_{selected_image}"
        selected_image_path = os.path.join(UPLOAD_FOLDER, selected_image_filename)
        shutil.copy(original_path, selected_image_path)

    # Handle uploaded file from local machine
    uploaded_file = request.files.get('uploaded_file')
    uploaded_image_path = None

    if uploaded_file and allowed_file(uploaded_file.filename):
        filename = secure_filename(uploaded_file.filename)
        uploaded_image_path = os.path.join(UPLOAD_FOLDER, filename)
        uploaded_file.save(uploaded_image_path)

    # Convert the paths to be relative to the 'static' folder
    selected_image_url = os.path.basename(selected_image_path)#None
    uploaded_image_url =os.path.basename(uploaded_image_path)# None
    '''
    if selected_image_path:
      print('selected_image_path',selected_image_path)
      selected_image_url = url_for('static', filename=f"uploads/{os.path.basename(selected_image_path)}")

    if uploaded_image_path:
      print('uploaded_image_path',uploaded_image_path)
      uploaded_image_url = url_for('static', filename=f"uploads/{os.path.basename(uploaded_image_path)}")
    '''
    print('selected_image_url',selected_image_url)
    print('uploaded_image_url',uploaded_image_url)
    TestData="/content/drive/My Drive/VDress/realtime_data/test/cloth"
    TestData2="/content/drive/My Drive/VDress/realtime_data/test/image"
    shutil.copy(UPLOAD_FOLDER+"/"+selected_image_url, TestData)
    shutil.copy(UPLOAD_FOLDER+"/"+uploaded_image_url, TestData2)
    time.sleep(10)
    virtual_result = main()
    print(virtual_result)
    return render_template('result.html',
                           selected_image=selected_image_url,
                           uploaded_image=uploaded_image_url)

print(f"To acces the Gloable link please click {public_url}")
#if __name__ == '__main__':
app.run(port=port_no)






